In [ ]:
# -*- coding: utf-8 -*-
"""
=====================================================================
 FASE 1 - Estudo de serie temporal do mini indice (WIN) em M1
=====================================================================
O que este script faz:
  1) Conecta no MetaTrader 5 e baixa OHLC de 1 minuto
  2) Reconstroi as 4 EMAs (100/200/300/400) igual ao seu grafico
  3) Estatisticas dos retornos de 1 min (distribuicao, autocorrelacao)
  4) Expoente de Hurst do preco (tendencia x reversao x aleatorio)
  5) Spread (preco - EMA300): AR(1), meia-vida e comparacao com o
     benchmark mecanico de passeio aleatorio  <-- o teste que importa
  6) Estados (abaixo das medias / na banda / acima): ocupacao,
     duracao media e "quando entra na banda vindo de baixo,
     pra onde ele sai?"
  7) Salva graficos PNG + CSV processado para a Fase 2

Requisitos (Windows, com o terminal MT5 ABERTO e logado):
    pip install MetaTrader5 pandas numpy matplotlib statsmodels

Uso:
    python fase1_estudo_win.py
=====================================================================
"""

from datetime import datetime, timedelta
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # salva PNG sem abrir janela
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------
# CONFIGURACAO - ajuste aqui
# ---------------------------------------------------------------
# O script usa o primeiro simbolo que existir na sua corretora.
# "WIN$N" = serie continua (ideal p/ 6 meses). Se so tiver o
# contrato vigente (ex. WINQ26), reduza DIAS_HISTORICO p/ ~50.
# SYMBOL_CANDIDATES = ["WIN$N", "WIN$", "WINFUT", "WINQ26"]
SYMBOL_CANDIDATES = ["WIN$"]
DIAS_HISTORICO    = 180
EMA_PERIODS       = [100, 200, 300, 400]
EMA_REF           = 300     # media de referencia do spread (onde voce vende)
ATR_PERIOD        = 20      # barras p/ normalizar pela volatilidade
WARMUP            = 2000    # barras descartadas p/ as EMAs "esquentarem"
SALVAR_CSV        = True
OUTDIR            = "saida_fase1"

LINHA = "-" * 64


# ---------------------------------------------------------------
# 1) DADOS
# ---------------------------------------------------------------
def baixar_dados_mt5(candidatos, dias):
    """Conecta no terminal MT5 local e baixa OHLC M1."""
    import MetaTrader5 as mt5  # import aqui: pacote so existe no Windows

    if not mt5.initialize():
        raise RuntimeError(
            f"Falha ao conectar no MT5: {mt5.last_error()}\n"
            "-> Abra o terminal MetaTrader 5, faca login e rode de novo."
        )

    simbolo = None
    for s in candidatos:
        if mt5.symbol_info(s) is not None:
            simbolo = s
            mt5.symbol_select(s, True)
            break

    if simbolo is None:
        disponiveis = [s.name for s in (mt5.symbols_get("WIN$*") or [])]
        mt5.shutdown()
        raise RuntimeError(
            "Nenhum dos simbolos candidatos existe nesta corretora.\n"
            f"Simbolos com 'WIN' disponiveis: {disponiveis}\n"
            "-> Copie o nome certo para SYMBOL_CANDIDATES no topo do script."
        )

    ate = datetime.now()
    desde = ate - timedelta(days=dias)
    rates = mt5.copy_rates_range(simbolo, mt5.TIMEFRAME_M1, desde, ate)
    mt5.shutdown()

    if rates is None or len(rates) == 0:
        raise RuntimeError(
            "O MT5 nao retornou dados. Dicas:\n"
            "  - Abra um grafico M1 do simbolo e espere o historico carregar\n"
            "  - Aumente 'Max. de barras no grafico' em Ferramentas > Opcoes > Graficos\n"
            "  - Reduza DIAS_HISTORICO"
        )

    df = pd.DataFrame(rates)
    df["time"] = pd.to_datetime(df["time"], unit="s")
    df = df.set_index("time")[["open", "high", "low", "close", "tick_volume"]]
    return simbolo, df


# ---------------------------------------------------------------
# 2) INDICADORES
# ---------------------------------------------------------------
def preparar(df):
    """EMAs, ATR, spread e classificacao de estado."""
    df = df.copy()

    # EMA identica a do MT5/Profit: alpha = 2/(N+1), recursiva
    for n in EMA_PERIODS:
        df[f"ema{n}"] = df["close"].ewm(span=n, adjust=False).mean()

    # ATR simples (media movel do True Range) so p/ normalizar escala
    tr = pd.concat(
        [
            df["high"] - df["low"],
            (df["high"] - df["close"].shift()).abs(),
            (df["low"] - df["close"].shift()).abs(),
        ],
        axis=1,
    ).max(axis=1)
    df["atr"] = tr.rolling(ATR_PERIOD).mean()

    # Spread em pontos e em unidades de ATR
    df["spread"] = df["close"] - df[f"ema{EMA_REF}"]
    df["spread_atr"] = df["spread"] / df["atr"]

    # Estado em relacao ao "pacote" de medias
    cols = [f"ema{n}" for n in EMA_PERIODS]
    ema_min = df[cols].min(axis=1)
    ema_max = df[cols].max(axis=1)
    df["estado"] = np.select(
        [df["close"] < ema_min, df["close"] > ema_max],
        ["abaixo", "acima"],
        default="banda",
    )

    # Descarta o aquecimento das EMAs
    warm = WARMUP if len(df) > 3 * WARMUP else len(df) // 4
    return df.iloc[warm:].dropna(subset=["atr"])


# ---------------------------------------------------------------
# 3) FUNCOES ESTATISTICAS
# ---------------------------------------------------------------
def half_life_ar1(spread):
    """
    Ajusta s_t = c + phi*s_{t-1} + e por MQO.
    Retorna (phi, erro-padrao de phi, meia-vida em barras).
    Meia-vida: tempo p/ um desvio cair pela metade = ln(0.5)/ln(phi).
    """
    s = pd.Series(spread).dropna().to_numpy(dtype=float)
    y, x = s[1:], s[:-1]
    res = sm.OLS(y, sm.add_constant(x)).fit()
    phi, se = res.params[1], res.bse[1]
    hl = np.log(0.5) / np.log(phi) if 0.0 < phi < 1.0 else np.nan
    return phi, se, hl


def hurst_exponent(precos, max_lag=120):
    """
    Hurst pela escala de difusao: std(p_{t+tau} - p_t) ~ tau^H.
      H < 0.5 -> reversao a media | H = 0.5 -> aleatorio | H > 0.5 -> tendencia
    Calculado sobre o log-preco.
    """
    p = np.log(pd.Series(precos).dropna().to_numpy(dtype=float))
    lags = np.unique(np.logspace(np.log10(2), np.log10(max_lag), 20).astype(int))
    tau = [np.std(p[lag:] - p[:-lag]) for lag in lags]
    H = np.polyfit(np.log(lags), np.log(tau), 1)[0]
    return H


def resumo_corridas(estados):
    """
    Agrupa barras consecutivas no mesmo estado ("corridas").
    Retorna (tabela de corridas, duracao media por estado).
    """
    grupos = (estados != estados.shift()).cumsum().rename("grupo")
    corridas = (estados.groupby(grupos)
                .agg(estado="first", duracao="size")
                .reset_index(drop=True))
    dur_media = corridas.groupby("estado")["duracao"].mean()
    return corridas, dur_media


def resolucao_da_banda(corridas):
    """
    Para cada corrida na banda: de onde veio -> pra onde saiu.
    E o esqueleto direcional da sua estrategia (sem alvo/stop/custos).
    """
    seq = corridas["estado"].tolist()
    contagem = {}
    for i in range(1, len(seq) - 1):
        if seq[i] == "banda":
            chave = (seq[i - 1], seq[i + 1])
            contagem[chave] = contagem.get(chave, 0) + 1
    return contagem


# ---------------------------------------------------------------
# 4) ANALISE + RELATORIO
# ---------------------------------------------------------------
def rodar_analise(df_bruto, simbolo):
    os.makedirs(OUTDIR, exist_ok=True)
    df = preparar(df_bruto)

    if len(df) == 0:
        raise RuntimeError("Dataframe vazio apos preparacao - confira se o MT5 "
                           "retornou OHLC valido (colunas open/high/low/close).")
    if len(df) < 5000:
        print(f"AVISO: apenas {len(df)} barras apos aquecimento - "
              "as estatisticas ficam fracas; puxe mais historico se possivel.")

    sessoes = pd.unique(df.index.date)
    print(LINHA)
    print(f"DADOS  |  {simbolo}  M1")
    print(LINHA)
    print(f"Barras apos aquecimento : {len(df):,}")
    print(f"Periodo                 : {df.index[0]}  ->  {df.index[-1]}")
    print(f"Sessoes (dias)          : {len(sessoes)}")

    # ---- Retornos de 1 minuto (em pontos) --------------------------------
    ret = df["close"].diff().dropna()
    print(LINHA)
    print("RETORNOS DE 1 MIN (em pontos)")
    print(LINHA)
    print(f"Media   : {ret.mean():+8.2f}")
    print(f"Desvio  : {ret.std():8.2f}")
    print(f"Assimetria (skew): {ret.skew():+6.2f}   Curtose: {ret.kurt():6.1f} "
          f"(normal = 0 -> caudas gordas se >> 0)")
    acs = [ret.autocorr(lag=k) for k in range(1, 6)]
    print("Autocorrelacao lags 1..5:", "  ".join(f"{a:+.3f}" for a in acs))
    print("  (lag-1 negativo em M1 e comum: ruido de microestrutura/bid-ask)")

    # ---- Hurst ------------------------------------------------------------
    H = hurst_exponent(df["close"])
    print(LINHA)
    print("EXPOENTE DE HURST (log-preco, horizontes de 2 a 120 min)")
    print(LINHA)
    veredito = ("reversao a media" if H < 0.47
                else "tendencia/persistencia" if H > 0.53
                else "~aleatorio")
    print(f"H = {H:.3f}  ->  {veredito}")

    # ---- Spread vs benchmark mecanico --------------------------------------
    phi, se, hl = half_life_ar1(df["spread"])
    phi_rw = (EMA_REF - 1) / (EMA_REF + 1)          # passeio aleatorio vs EMA
    hl_rw = np.log(0.5) / np.log(phi_rw)
    z = (phi - phi_rw) / se

    print(LINHA)
    print(f"SPREAD  s = close - EMA{EMA_REF}")
    print(LINHA)
    print(f"Media do spread         : {df['spread'].mean():+8.1f} pts")
    print(f"Desvio do spread        : {df['spread'].std():8.1f} pts")
    print(f"phi estimado (AR1)      : {phi:.5f}   (ep {se:.5f})")
    hl_lo = np.log(0.5) / np.log(min(phi + 2 * se, 0.999999))
    hl_hi = np.log(0.5) / np.log(max(phi - 2 * se, 1e-6))
    print(f"Meia-vida empirica      : {hl:6.0f} barras (~min)   "
          f"[IC ~95%: {hl_hi:.0f} a {hl_lo:.0f} min]")
    print(f"Benchmark passeio aleat.: phi_RW = {phi_rw:.5f}  ->  meia-vida ~ {hl_rw:.0f} min")
    print(f"z = (phi - phi_RW)/ep   : {z:+.1f}")
    if z < -2:
        print(">> Reversao MAIS RAPIDA que a mecanica: indicio de reversao genuina.")
    elif z > 2:
        print(">> Reversao MAIS LENTA que a mecanica: indicio de persistencia -")
        print("   o preco 'foge' da media; cuidado ao vender pullback cedo demais.")
    else:
        print(">> Indistinguivel de um passeio aleatorio contra a propria EMA.")
    print("Obs.: benchmark assume incrementos independentes; e uma regua, nao veredito.")

    # ADF so como referencia didatica (quase sempre rejeita, por construcao)
    adf = adfuller(df["spread"].dropna(), maxlag=20, autolag="AIC")
    print(f"ADF do spread: estatistica {adf[0]:.1f}, p-valor {adf[1]:.4f} "
          "(rejeita raiz unitaria quase por construcao - por isso o benchmark acima)")

    # ---- Estados ------------------------------------------------------------
    corridas, dur_media = resumo_corridas(df["estado"])
    ocup = df["estado"].value_counts(normalize=True)

    print(LINHA)
    print("ESTADOS: abaixo das 4 medias / na banda / acima")
    print(LINHA)
    for e in ["abaixo", "banda", "acima"]:
        pct = 100 * ocup.get(e, 0.0)
        dm = dur_media.get(e, float("nan"))
        print(f"{e:>7}: {pct:5.1f}% do tempo | duracao media da visita: {dm:6.1f} min")

    res = resolucao_da_banda(corridas)
    de_baixo = {k: v for k, v in res.items() if k[0] == "abaixo"}
    tot = sum(de_baixo.values())
    print("\nQuando entra na BANDA vindo de BAIXO (o seu contexto):")
    if tot > 0:
        volta = de_baixo.get(("abaixo", "abaixo"), 0)
        rompe = de_baixo.get(("abaixo", "acima"), 0)
        print(f"  volta pra baixo : {volta:5d}  ({100*volta/tot:5.1f}%)  <- direcao da sua venda")
        print(f"  rompe pra cima  : {rompe:5d}  ({100*rompe/tot:5.1f}%)")
        print(f"  total de visitas: {tot}")
        print("  ATENCAO: num passeio aleatorio simulado, 'volta pra baixo' ja da ~75%")
        print("  (as medias perseguem o preco). O piso de comparacao e ~75%, nao 50%.")
        print("  (so direcao de saida da banda; alvo/stop/custos entram na Fase 2)")
    else:
        print("  Nenhuma ocorrencia no periodo (?) - confira os dados.")

    # ---- Graficos -------------------------------------------------------------
    _graficos(df, simbolo, sessoes)

    # ---- CSV p/ Fase 2 ---------------------------------------------------------
    if SALVAR_CSV:
        caminho = os.path.join(OUTDIR, f"dados_{simbolo.replace('$','S')}_M1.csv")
        df.to_csv(caminho)
        print(LINHA)
        print(f"CSV processado salvo em: {caminho}")

    print(LINHA)
    print("PROXIMA FASE: estudo de eventos - formalizar o setup (esticada +")
    print("retorno a EMA300), medir MFE/MAE, alvo-antes-do-stop, custos e")
    print("comparar com entradas aleatorias (bootstrap).")
    print(LINHA)
    return df


def _graficos(df, simbolo, sessoes):
    # (1) Ultima sessao com as 4 EMAs - p/ conferir com o Profit
    ultima = df[df.index.date == sessoes[-1]]
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(ultima.index, ultima["close"], lw=0.9, color="black", label="close")
    for n in EMA_PERIODS:
        ax.plot(ultima.index, ultima[f"ema{n}"], lw=1.1, label=f"EMA{n}")
    ax.set_title(f"{simbolo} M1 - ultima sessao ({sessoes[-1]}) - confira com o Profit")
    ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTDIR, "01_ultima_sessao_emas.png"), dpi=120)
    plt.close(fig)

    # (2) Spread das ultimas 5 sessoes
    mask = np.isin(df.index.date, sessoes[-5:])
    rec = df[mask]
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(rec.index, rec["spread"], lw=0.7)
    ax.axhline(0, color="red", lw=1)
    ax.set_title(f"Spread close - EMA{EMA_REF} (pts) - ultimas 5 sessoes")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTDIR, "02_spread_recente.png"), dpi=120)
    plt.close(fig)

    # (3) Distribuicao do spread normalizado por ATR
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.hist(df["spread_atr"].dropna(), bins=120, alpha=0.85)
    ax.axvline(0, color="red", lw=1)
    ax.set_title(f"Distribuicao do spread / ATR{ATR_PERIOD} (periodo todo)")
    ax.set_xlabel("desvios em unidades de ATR")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTDIR, "03_hist_spread_atr.png"), dpi=120)
    plt.close(fig)

    print(f"\nGraficos salvos na pasta '{OUTDIR}/'")


# ---------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------
def main():
    print(LINHA)
    print("FASE 1 - baixando dados do MT5...")
    simbolo, df = baixar_dados_mt5(SYMBOL_CANDIDATES, DIAS_HISTORICO)
    print(f"OK: {len(df):,} barras de {simbolo}")
    rodar_analise(df, simbolo)


if __name__ == "__main__":
    main()

----------------------------------------------------------------
FASE 1 - baixando dados do MT5...
OK: 68,552 barras de WIN$
----------------------------------------------------------------
DADOS  |  WIN$  M1
----------------------------------------------------------------
Barras apos aquecimento : 66,552
Periodo                 : 2026-01-13 14:12:00  ->  2026-07-06 18:31:00
Sessoes (dias)          : 119
----------------------------------------------------------------
RETORNOS DE 1 MIN (em pontos)
----------------------------------------------------------------
Media   :    +0.00
Desvio  :   104.45
Assimetria (skew):  +0.76   Curtose:  221.4 (normal = 0 -> caudas gordas se >> 0)
Autocorrelacao lags 1..5: -0.037  +0.009  +0.001  +0.006  +0.001
  (lag-1 negativo em M1 e comum: ruido de microestrutura/bid-ask)
----------------------------------------------------------------
EXPOENTE DE HURST (log-preco, horizontes de 2 a 120 min)
-----------------------------------------------------------